In [2]:
import pandas as pd
import numpy as np
import tqdm
from dotenv import load_dotenv
import litellm
from typing import Callable, Literal
from string import punctuation
from pydantic import BaseModel, Field
import asyncio
import os
import zipfile

In [3]:
from helpers import (
    # BM25 
    build_index, score_bm25, search_bm25,
    # Evaluation
    evaluate_search,
    # Embeddings
    get_local_model, batch_embed_local,
    # Similarity
    batch_cosine_similarity,
    # Utility
    normalize_scores
)

In [4]:
os.environ.get("OPENAI_API_KEY")
print(f"Length of OpenAI API key: {len(os.environ["OPENAI_API_KEY"])} characters")

Length of OpenAI API key: 164 characters


In [7]:
with zipfile.ZipFile("fordham-website-windows.zip", "r") as zip_ref:
    zip_ref.extractall("extracted_content")

In [ ]:
os.listdir("extracted_content")[:3]

['0001144e6d954f94682637e541ad5d7f.md',
 '000121f75daed3fee3eb14cdb934a788.md',
 '001fb84f7d97bdd24d99b581d920a602.md']

In [22]:
from sentence_transformers import SentenceTransformer


In [17]:
def load_fordham_markdown(zip_path: str = "fordham-website-windows.zip") -> pd.DataFrame:
    """
    Load Fordham website pages from a zip archive into a DataFrame.

    Returns a DataFrame with two columns:
      - 'url': first line of each markdown file
      - 'content': remaining markdown content
    """
    rows = []

    with zipfile.ZipFile(zip_path, "r") as z:
        # iterate over all files in the archive
        for name in z.namelist():
            # skip directories or non-markdown files
            if not name.endswith(".md"):
                continue

            with z.open(name, "r") as f:
                text = f.read().decode("utf-8", errors="ignore")
                lines = text.splitlines()

                if not lines:
                    continue

                url = lines[0].strip()
                # join the rest back into markdown content, stripping leading blank lines
                content = "\n".join(lines[1:]).lstrip("\n")

                rows.append({"url": url, "content": content})

    return pd.DataFrame(rows, columns=["url", "content"])

In [18]:
df_pages = load_fordham_markdown()
df_pages.head()

,url,content
0,https://www.fordham.edu/,## Doing Good That Becomes Greater As The Jesu...
1,https://www.fordham.edu/research,Uncovering the Limitless Possibilities of Scie...
2,https://www.fordham.edu/ccel,# Center for Community Engaged Learning\n\n###...
3,https://www.fordham.edu/fordham-college-at-lin...,# Fordham College at Lincoln Center\n\n## Why ...
4,https://www.fordham.edu/academics,# Academics\n\n## 8 schools. 3 campuses.\n\n![...


In [23]:
local_model = "all-MiniLM-L6-v2"
model = SentenceTransformer(local_model)

model.encode(df_pages.iloc[0]["content"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 151.63it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


array([ 9.54504684e-02, -5.49975932e-02,  2.24007145e-02, -5.60574755e-02,
       -1.36425998e-02, -4.27612988e-03, -1.04771569e-01, -5.68896681e-02,
        7.77073656e-05, -3.20745111e-02,  5.02321385e-02,  4.20842133e-02,
       -1.15366258e-01, -4.16201819e-03, -4.22632545e-02,  2.95201782e-03,
       -4.54273894e-02,  9.16817784e-03, -1.19019281e-02,  1.62669253e-02,
        2.23941263e-02, -5.75443134e-02,  3.08574066e-02,  6.39920011e-02,
       -1.55548053e-02,  9.75854546e-02, -2.06553712e-02, -5.37699759e-02,
        1.62028950e-02, -1.69699602e-02, -4.71134409e-02,  4.01875935e-02,
        5.42141423e-02, -1.17278937e-02,  6.06362754e-03,  7.08618164e-02,
        1.22157551e-01,  3.11700795e-02,  1.06697753e-01, -2.68105272e-04,
        2.26548109e-02, -5.29386289e-02,  5.12272269e-02,  9.03758034e-02,
       -5.74289747e-02, -3.22188884e-02,  1.50580890e-03,  2.98912339e-02,
        5.59889944e-03, -5.50224446e-03, -2.19483413e-02, -4.54106778e-02,
        7.54156187e-02, -

In [24]:
def chunk_text(text: str, max_chars: int = 5000) -> list[str]:
    """
    Split a long text into consecutive, contiguous, non-overlapping chunks,
    each of at most `max_chars` characters.

    Args:
        text: The full page content as a string.
        max_chars: Maximum number of characters per chunk.

    Returns:
        List of chunk strings (order preserved).
    """
    if text is None:
        return []

    text = str(text)
    chunks = []
    for start in range(0, len(text), max_chars):
        chunks.append(text[start:start + max_chars])
    return chunks

In [25]:
chunks_df = (
    df_pages
    .assign(chunks=lambda df: df["content"].apply(chunk_text))
    .explode("chunks", ignore_index=True)
    .rename(columns={"chunks": "chunk"})
)

In [29]:
chunks_df.columns

Index(['url', 'content', 'chunk'], dtype='str')

In [30]:
len(os.listdir("extracted_content"))

9560

In [31]:
len(chunks_df)

13950

In [33]:
len(chunks_df.index)

13950

In [36]:
len(chunks_df["url"].unique())

9553

In [37]:
from helpers import snowball_tokenize

In [40]:
for chunk in tqdm.tqdm(chunks_df["chunk"], desc = "Chunk embeddings"):
    snowball_tokenize(chunk)
    model.encode(df_pages.iloc[0]["content"])

Chunk embeddings:   1%|          | 111/13950 [00:19<41:27,  5.56it/s] 


KeyboardInterrupt: 

In [ ]:
# Gemini

# Pre-calculating for speed
n_batches = 14
chunks_df['batch_id'] = range(len(chunks_df))
chunks_df['batch_id'] %= n_batches

# Now you can group by it
for batch_id, data in chunks_df.groupby('batch_id'):
    for chunk
        snowball_tokenize(chunk)
        model.encode(df_pages.iloc[0]["content"])
    print(f"Processing batch {batch_id}...")

Processing batch 0...
Processing batch 1...
Processing batch 2...
Processing batch 3...
Processing batch 4...
Processing batch 5...
Processing batch 6...
Processing batch 7...
Processing batch 8...
Processing batch 9...
Processing batch 10...
Processing batch 11...
Processing batch 12...
Processing batch 13...


In [52]:
chunks_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 13950 entries, 0 to 13949
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   url       13950 non-null  str  
 1   content   13950 non-null  str  
 2   chunk     13950 non-null  str  
 3   batch_id  13950 non-null  int64
dtypes: int64(1), str(3)
memory usage: 436.1 KB


In [55]:
def batch_logic(subset_df):
    # Perform a calculation on the whole batch at once
    subset_df["chunk_embed"] = model.encode(
        snowball_tokenize(
            subset_df["chunk"]
            )
        )
    return subset_df


In [56]:

# Process each batch and store results
processed_batches = [batch_logic(data) for _, data in chunks_df.groupby('batch_id')]
final_df = pd.concat(processed_batches)

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

In [63]:
chunks_df["chunk_tknz"] = chunks_df["chunk"].copy()


In [65]:
for batch_id, batch_df in chunks_df.groupby('batch_id'):
    print(f"--- Starting Batch {batch_id} ---")
    
    # Efficiently iterate over rows in this batch
    for row in tqdm.tqdm(batch_df.itertuples(index=True), desc = "Row tokenizations"):
        # Access data using row.column_name
        snowball_tokenize(row.chunk_tknz)

--- Starting Batch 0 ---


Row tokenizations: 997it [00:08, 117.56it/s]


--- Starting Batch 1 ---


Row tokenizations: 997it [00:00, 1313.07it/s]


--- Starting Batch 2 ---


...

Row tokenizations: 996it [00:00, 1193.70it/s]


--- Starting Batch 12 ---


Row tokenizations: 996it [00:01, 754.79it/s] 


--- Starting Batch 13 ---


Row tokenizations: 996it [00:00, 1640.90it/s]


In [68]:
for batch_id, batch_df in chunks_df.groupby('batch_id'):
    print(f"--- Starting Batch {batch_id} ---")
    
    # Efficiently iterate over rows in this batch
    for row in tqdm.tqdm(batch_df.itertuples(index=True), desc = "Row embeddings"):
        # Access data using row.column_name
        model.encode(row.chunk_tknz)

--- Starting Batch 0 ---


Row embeddings: 997it [01:48,  9.17it/s]


--- Starting Batch 1 ---


Row embeddings: 997it [01:22, 12.08it/s]


--- Starting Batch 2 ---


...

Row embeddings: 996it [01:21, 12.16it/s]


--- Starting Batch 12 ---


Row embeddings: 996it [01:22, 12.09it/s]


--- Starting Batch 13 ---


Row embeddings: 996it [01:20, 12.36it/s]


In [70]:
chunks_df.iloc[0]["chunk_tknz"]

'## Doing Good That Becomes Greater As The Jesuit University of New York\n\nWe’re located in New York City—driven by our Jesuit values and tackling today’s most pressing issues at the center of the world stage.\n\n## We Are Leaders, Dreamers, Achievers, And Doers\n\nWith sound hearts, strong minds, and the wisdom to take charge, generations of Rams have found what they have needed to grow—the opportunities, connections, and support of this community.\n\n## From Winding Elms to Bustling City Blocks\n\nWith residential campuses in the Bronx and Manhattan, as well as campuses in Westchester and London, Fordham provides endless opportunities to start working toward your career and building the life you want.\n\n\n## We’re Drawn to Where We’re Needed Most\n\nExplore how our values come to life: how Fordham’s students, faculty, and alumni contribute to society and make lives better.\n\n**Notice of Nondiscriminatory Policy:**\n\nFordham University admits students of any race, color, national 